In [ ]:
%pip install -q sentence-transformers

In [ ]:
import os
import gc
import re
import pandas as pd
import numpy as np
import torch
from sentence_transformers import CrossEncoder

In [ ]:
import re

def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        u"\U0001FA00-\U0001FA6F"
        u"\U00002500-\U00002BFF"
        u"\U00002600-\U000026FF"
        u"✀-➿"
        u"⌀-⏿"
        "✓✗⚠►▸▼▲◄→←↑↓│├└┌┐┘┤┬┴┼"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)


def replace_arrows(text):
    if not isinstance(text, str):
        return ''
    arrows = [
        ('-->', ' leads to '),
        ('=>',  ' leads to '),
        ('→',   ' leads to '),
        ('►',   ' leads to '),
        ('▼',   ' then '),
        ('▲',   ' then '),
        ('←',   ' comes from '),
    ]
    for arrow, replacement in arrows:
        text = text.replace(arrow, replacement)
    return text


def extract_code_block_content(text):
    def clean_block(match):
        code = match.group(1)
        code = ''.join(
            ' ' if (0x2500 <= ord(ch) <= 0x257F) else ch
            for ch in code
        )
        code = re.sub(r'[ \t]+', ' ', code)
        code = re.sub(r'\n{2,}', '\n', code)
        return code.strip()

    return re.sub(
        r'```(?:\w+)?\n?(.*?)```',
        clean_block,
        text,
        flags=re.DOTALL
    )


def convert_table_to_text_safe(text):
    lines = text.split('\n')
    result = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('|') and line.endswith('|'):
            if re.match(r'^\|[\s\-:]+\|$', line):
                i += 1
                continue

            cells = [c.strip() for c in line.strip('|').split('|') if c.strip()]

            if i + 1 < len(lines) and re.match(r'^\|[\s\-:|]+\|$', lines[i + 1].strip()):
                headers = cells
                i += 2

                while i < len(lines):
                    row = lines[i].strip()
                    if not (row.startswith('|') and row.endswith('|')):
                        break

                    values = [c.strip() for c in row.strip('|').split('|') if c.strip()]
                    headers = [re.sub(r'\*+(.+?)\*+', r'\1', h) for h in headers]
                    values  = [re.sub(r'\*+(.+?)\*+', r'\1', v) for v in values]

                    pairs = []
                    for h, v in zip(headers, values):
                        if h and v:
                            pairs.append(f"{h} is {v}")

                    if pairs:
                        result.append('. '.join(pairs) + '.')

                    i += 1
            else:
                result.append('. '.join(cells) + '.')
                i += 1
        else:
            result.append(lines[i])
            i += 1

    return '\n'.join(result)


def ensure_line_terminators(text):
    lines = text.split('\n')
    result = []
    for line in lines:
        stripped = line.strip()
        match = re.match(r'^(\-|\*|•|\d+[\.\)])\s+(.+)', stripped)
        if match:
            stripped = match.group(2).strip()
        if stripped and stripped[-1] not in '.!?:':
            stripped += '.'
        result.append(stripped)
    return '\n'.join(result)


def ensure_sentence_terminators(text):
    paragraphs = text.split('\n')
    result = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            result.append('')
            continue
        doc = nlp(para)
        sentences = []
        for sent in doc.sents:
            s = sent.text.strip()
            if s and s[-1] not in '.!?:':
                s += '.'
            sentences.append(s)
        result.append(' '.join(sentences))
    return '\n'.join(result)

def remove_markdown_dividers(text):
    return re.sub(r'^\s*[-_*]{2,}\s*$', '', text, flags=re.MULTILINE)




def clean_markdown(text):



    text = replace_arrows(text)
    text = remove_emoji(text)

    text = extract_code_block_content(text)

    text = convert_table_to_text_safe(text)

    text = remove_markdown_dividers(text)




    text = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
    text = re.sub(r'^[\-\*\+]\s+', '', text, flags=re.MULTILINE)



    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")
    text = re.sub(r'"{2,}', '"', text)
    text = text.replace('"', '')
    text = text.replace('—', ' ')
    text = text.replace('–', ' ')
    text = re.sub(r'[ \t]+', ' ', text)  

    text = re.sub(r'#+\s*', '', text)
   


    return text.strip()

In [ ]:
INPUT_DIR  = "../Dataset/responses"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Working dir:', os.getcwd())
print('Data dir   :', INPUT_DIR)
print('Output dir :', OUTPUT_DIR)

In [ ]:
LABEL_MAP = ['contradiction', 'entailment', 'neutral']
nli_model = CrossEncoder("dleemiller/ModernCE-large-nli")
print("ModernCE-large-nli loaded")

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


def run_nli_batch(model, premise_list, hypothesis_list, batch_size=16):
    """Run NLI on a list of (premise, hypothesis) pairs in one direction."""
    pairs = list(zip(premise_list, hypothesis_list))
    all_results = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        logits = model.predict(batch)
        for logit in logits:
            probs = softmax(logit)
            label = LABEL_MAP[probs.argmax()]
            all_results.append({
                'label':         label,
                'entailment':    round(float(probs[1]), 4),
                'neutral':       round(float(probs[2]), 4),
                'contradiction': round(float(probs[0]), 4),
            })
    return all_results


def run_bidirectional_nli_for_file(file_path, output_path):
   
    df = pd.read_csv(file_path, sep='\t')

    if 'Answer_Model' in df.columns and 'Model_Name' not in df.columns:
        df['Model_Name'] = df['Answer_Model']

    df['LLM_Response']   = df['LLM_Response'].astype(str)
    df['Clean_Response'] = df['LLM_Response'].apply(clean_markdown)

    print(f"\n Loaded {len(df)} rows from {file_path}")
    print(f"   Models: {df['Model_Name'].unique()}")
    print(f"   Levels: {df['Literacy_Level'].unique()}")

    nli_results = []
    compare_levels = ['Mid Literacy', 'Low Literacy']  

    for model_name in df['Model_Name'].unique():
        df_model = df[df['Model_Name'] == model_name]

        hl_df = (df_model[df_model['Literacy_Level'] == 'High Literacy']
                 [['ID', 'Clean_Response']]
                 .rename(columns={'Clean_Response': 'HL_Text'}))
        if hl_df.empty:
            print(f"⚠ No High Literacy rows for: {model_name}")
            continue

        for level in compare_levels:
            comp_df = (df_model[df_model['Literacy_Level'] == level]
                       [['ID', 'Clean_Response']]
                       .rename(columns={'Clean_Response': 'Comp_Text'}))
            pairs = pd.merge(hl_df, comp_df, on='ID')
            if pairs.empty:
                print(f"{model_name} - {level}: no pairs, skipping")
                continue

            fwd = run_nli_batch(
                nli_model,
                pairs['HL_Text'].tolist(),
                pairs['Comp_Text'].tolist(),
                batch_size=16,
            )
            rev = run_nli_batch(
                nli_model,
                pairs['Comp_Text'].tolist(),
                pairs['HL_Text'].tolist(),
                batch_size=16,
            )

            for i, (_, row) in enumerate(pairs.iterrows()):
                f, r = fwd[i], rev[i]
                nli_results.append({
                    'Model_Name':         model_name,
                    'ID':                 row['ID'],
                    'Compared_Level':     level,
                    'fwd_label':          f['label'],
                    'fwd_entailment':     f['entailment'],
                    'fwd_neutral':        f['neutral'],
                    'fwd_contradiction':  f['contradiction'],
                    'rev_label':          r['label'],
                    'rev_entailment':     r['entailment'],
                    'rev_neutral':        r['neutral'],
                    'rev_contradiction':  r['contradiction'],
                    'avg_entailment':     round((f['entailment']    + r['entailment'])    / 2, 4),
                    'avg_neutral':        round((f['neutral']       + r['neutral'])       / 2, 4),
                    'avg_contradiction':  round((f['contradiction'] + r['contradiction']) / 2, 4),
                })
            print(f"{model_name} | {level} ↔ HL — {len(pairs)} bidirectional pairs done")
            gc.collect()

    out_df = pd.DataFrame(nli_results)
    out_df.to_csv(output_path, sep='\t', index=False)
    print(f"\n Saved: {output_path}")

    summary = out_df.groupby(['Model_Name', 'Compared_Level']).agg(
        Pairs              = ('fwd_label',         'count'),
        Avg_Entailment     = ('avg_entailment',    'mean'),
        Avg_Neutral        = ('avg_neutral',       'mean'),
        Avg_Contradiction  = ('avg_contradiction', 'mean'),
        Fwd_Entailment     = ('fwd_entailment',    'mean'),
        Rev_Entailment     = ('rev_entailment',    'mean'),
    ).round(4)
    print(f"\n{'='*90}")
    print(f"  BIDIRECTIONAL NLI SUMMARY  ({file_path})")
    print(f"{'='*90}")
    print(summary.to_string())

    return out_df

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_claude_explicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_gemini_explicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_gpt_explicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_llama_explicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_qwen_explicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_claude_implicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_gemini_implicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_gpt_implicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_llama_implicit.tsv"),
)

In [ ]:
run_bidirectional_nli_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "NLI_bidirectional_qwen_implicit.tsv"),
)